# ☢️ HW2 — Can You Finish This Reactor Model?
## Completing a half-built PyTorch architecture to keep the basin from overheating

> **The story.** You've just been made **manager of the reactor's main basin**. Its temperature
> must be watched every second — but the real sensors are failing under radiation, so the plant
> relies on a **backup predictor**: a neural network that estimates the basin temperature from an
> array of *secondary sensors* and a *thermal camera*.
>
> The engineer who built it was a gem — he nailed the data pipeline and even designed the whole
> model architecture, leaving a confident note: *"it will work."* Then, burned out (and never
> granted that raise **you** denied him), he **quit mid-build**. You inherit a pipeline plan with
> several sub-modules **left unfinished** — and a couple of an intern's mistakes baked in.
>
> Your job isn't to redesign anything. It's to **read the plan and finish each block correctly**,
> so the predictor runs and flags an overheat before the basin boils.

**How this notebook works**
- You'll open the **pipeline plan**, then complete each unfinished block — marked **🎯**.
- The sub-modules are **independent**: do them in any order, each with its own quick self-check.
- A few **diagrams & quizzes** build the intuition before you touch the code.
- The encoder, the signal module and the training loop are **imported, not shown** — you open each
  only when a task leads you to it, just like inheriting real code.


## 0. Setup

This notebook is **self-contained**: the first cell pulls the rest of the exercise files (the
`basin_lab.py` model blocks, the `basin_viz.py` diagrams, and the `basin_data.py` generator)
directly from the course repository. Run the setup cells below in order.

> ✅ The course repo is **public**, so no GitHub token is needed — the cell clones it directly.
> If you're running this notebook from inside a local clone of the repo, the setup cell skips the
> clone and just uses the files already on disk.

> 💻 **GPU runtime (optional).** Everything here is tiny and runs in seconds on CPU. If you want a
> GPU anyway: *Runtime → Change runtime type → T4 GPU*.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Locate this homework folder (holds helpers/ + data/) and make the helpers importable.
_HW = os.path.join("workshop", "homework")
for _root in [os.path.join(REPO_NAME, _HW), ".", _HW,
              os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "helpers", "basin_lab.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the homework folder (workshop/homework with helpers/ + data/). "
        "On Colab, re-run this cell to clone it; locally, run the notebook from inside a clone of the FDD-WE0-public repo.")
sys.path.insert(0, os.path.join(os.getcwd(), "helpers"))   # make the helpers importable
print("Working directory:", os.getcwd())


**0.2 — Install dependencies.** Torch is already on Colab; this just adds `einops` and pins
the rest.

In [ ]:
%pip install -q -r requirements_block_basin.txt

**0.3 — Import the libraries** and the inherited model code. The given building blocks live
in **`basin_lab`** (the frozen encoder, the signal module, the training loop) and the diagrams &
quizzes in **`basin_viz`**.

In [ ]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from einops import rearrange, reduce

import basin_data
import basin_lab
import basin_viz
importlib.reload(basin_data); importlib.reload(basin_lab); importlib.reload(basin_viz)

# 🔒 Seed every RNG so the run is reproducible — same weights, same numbers every time.
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())

---
# Part 1 — Open the pipeline plan

The first thing on your desk is the engineer's **architecture plan**. Read it before touching
anything: it shows **two input branches** that get **fused**, then turned into a single
temperature.

In [ ]:
basin_viz.pipeline_overview()

Take a moment with that plan — it's your map for the whole notebook.

- The **thermal branch**: camera frames → a *reshape* → the **frozen vision encoder** → features.
- The **signal branch**: the secondary-sensor time series → a 1-D-CNN feature extractor.
- **Fusion**: the two feature vectors are concatenated, sent through an **MLP core**, and a
  **regression head** produces the basin temperature in °C.

### 🧠 Quick check — did you read the plan right?

In [ ]:
basin_viz.true_false_quiz("pipeline_reading")

### Load the backup-system data

The engineer's data pipeline is done, so we just load it. Each sample has **two modalities** and a
temperature target (standardized — more on that when we train).

In [ ]:
bundle = basin_data.load_basin()
print("sensor series :", bundle["X_sensor"].shape, " (batch, 4 channels, 64 timesteps)")
print("thermal frames:", bundle["X_thermal"].shape, " (batch, 3 frames, 16, 16, 1 channel)")
print("target        :", bundle["y"].shape, " basin temperature, standardized")
print(f"\ntemperature ranges ~{bundle['temp_c'].min():.0f}–{bundle['temp_c'].max():.0f} °C,"
      f" overheat line at {bundle['overheat_c']:.0f} °C")

train_loader, val_loader = basin_lab.make_loaders(bundle, batch_size=128)
device = basin_lab.pick_device()
print("Training on:", device)

---
# Part 2 — Three quick warm-ups

The unfinished blocks use three moves. Let's make sure each is concrete on a tiny example *before*
you wire it into the model — no need to guess later.

### Warm-up 1 — reactivate einops, the way you saw it in notebook 05

Recall the rule: a pattern is a little sentence, `"input axes -> output axes"`. Two moves come
back here:
- **Reordering** the names on the right *permutes* the axes (no numbers move, just their layout).
- **`reduce`** collapses any axis you drop from the output — averaging (or max-ing) it away.

These two tiny examples are just to dust the syntax off — *not* the thermal reshape itself (you'll
write that one yourself, and it chains both moves at once).

In [ ]:
# permute: swap the last two axes of a 3-D tensor — e.g. (batch, time, channels) -> (batch, channels, time)
seq = torch.arange(2 * 3 * 4).reshape(2, 3, 4)
print("permute  'b t c -> b c t':", tuple(rearrange(seq, "b t c -> b c t").shape))   # (2, 4, 3)

# reduce: average a whole axis away (here the trailing one)
feat = torch.arange(2 * 3 * 4, dtype=torch.float32).reshape(2, 3, 4)
print("reduce   'b c d -> b c' (mean):", tuple(reduce(feat, "b c d -> b c", "mean").shape))   # (2, 3)

### Warm-up 2 — what a 1-D convolution actually *looks for*

The signal branch is built from `Conv1d` layers — the 1-D cousin of the image convolutions you
saw on pictures. First, **the pure mechanic**: a convolution slides a small **kernel** along the
signal, and at each step it multiplies the aligned values, adds them into one number, then shifts
right. The cell below shows that step by step with a trivial kernel.

In [ ]:
basin_viz.conv1d_demo()

Now the interesting part: *which numbers are in the kernel* decides **what pattern it
responds to** — just like a 2-D kernel can pick out edges in an image. Below, the **same** signal
is run through **two** kernels. Form a guess before the quiz — don't compute anything, just look at
where each output is large versus near zero.

In [ ]:
basin_viz.conv1d_kernel_gallery()

### 🧠 Name the kernels

In [ ]:
basin_viz.mc_quiz("conv1d_kernels")

And the real thing — a single `nn.Conv1d` carrying Kernel A's weights `[1, 0, −1]` does
exactly this arithmetic on its own little signal:

In [ ]:
sig = torch.tensor([[[2., 1, 3, 5, 4, 2]]])      # (batch=1, channels=1, length=6)
conv = nn.Conv1d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    conv.weight.copy_(torch.tensor([[[1., 0, -1]]]))  # Kernel A, the slope detector
print("Conv1d output:", conv(sig).squeeze().tolist())

### Warm-up 3 — a module is an ordered list of children

PyTorch modules **nest**: a module is built from child modules, registered **in order**. You read
them with `.children()` — but that hands back a *one-shot iterator*, not a list, so you can't index
or slice it directly. Wrap it in `list(...)` once and you get an ordinary Python list you can index,
slice, and reorder. That's the whole trick behind re-using part of someone else's network.

To feel that the order is real, let's **swap two same-shape blocks** and watch the output change.

In [ ]:
demo = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(),
    nn.Linear(8, 8),          # child 2  ── same shape (8→8) as child 4 …
    nn.ReLU(),
    nn.Linear(8, 8),          # child 4  ── … so the two are interchangeable in shape
    nn.Linear(8, 2),
)
print(demo)

children = list(demo.children())     # list(...) so we can index/slice — .children() alone is an iterator
print("\nchildren in order:", [type(c).__name__ for c in children])

x = torch.randn(1, 4)
before = demo(x)
demo[2], demo[4] = demo[4], demo[2]  # swap the two Linear(8,8) blocks in place — shapes still line up
after = demo(x)
print("same shape after the swap?", tuple(before.shape) == tuple(after.shape),
      "| but identical numbers?", torch.allclose(before, after))
print("→ order matters: the model still runs, but the result changed.")

---
# Part 3 — Finish the unfinished blocks

Four 🎯 tasks, each self-checking on a tiny batch so you always know it works before moving on.
The **first three** build the branches and the core — they're independent, so do them in any
order. **Block 4 assembles them**, so it comes last.

## 🎯 Block 1 — the signal feature extractor

The plan imports **`SpectroNet`**, a generic off-the-shelf 1-D signal network. There's a catch the
engineer noted: *its `forward` ends in a classifier* — it outputs class logits, not the features we
want to fuse. Open it and see.

In [ ]:
basin_viz.show_source(basin_lab.SpectroNet)

In [ ]:
basin_viz.spectronet_diagram()

### 🧠 Which move gives us the features?

In [ ]:
basin_viz.mc_quiz("spectronet_extract")

### 🎯 Build the extractor

Use the **warm-up 3** move. First turn the children into a list (so you can slice it), then build a
fresh `nn.Sequential` from **every child except the last** — that drops the `classifier` and leaves
a module that stops at the `(B, 32)` features.

> 💡 `nn.Sequential` wants the layers as *separate arguments*, not one list — so spread the sliced
> list in with a leading `*`, the **unpacking operator**: it hands each item of the list to
> `nn.Sequential` as its own argument (e.g. `nn.Sequential(*some_list)`).

In [ ]:
spectro = basin_lab.SpectroNet(in_channels=4)

children = list(spectro.children())        # list() so we can slice it (warm-up 3)
# 🎯 build a Sequential from every child EXCEPT the last; spread them in with a leading *
signal_extractor = nn.Sequential( ??? )    # 🎯 take all but the final item of `children`, then *spread it*

# --- self-check: features, not class logits ---
out = signal_extractor(torch.zeros(2, 4, 64))
print("extractor output:", tuple(out.shape))
assert out.shape == (2, basin_lab.SIGNAL_FEATURES), "Should be (2, 32) features, not (2, 5) logits."
print("✅ signal branch outputs 32 features")

## 🎯 Block 2 — reshape the thermal frames for the encoder

The frozen **vision encoder** expects one batch of single-channel images, shaped
**`(batch, 1, 16, 16)`** — channels *first*. But our thermal data arrives as
**`(batch, 3 frames, 16, 16, 1)`** — channels *last*, with a frames axis. The engineer's note:
*"reshape the image batches before forwarding them into the vision encoder."*

The plan (you write the right-hand side of each `rearrange`/`reduce`):
1. starting from `(b, f, h, w, c)`, **fold the frames into the batch** *and* move the channel axis
   to the front, so the encoder can run on every frame at once;
2. run the encoder → one feature vector per frame;
3. starting from `(b, f, d)`, **average over the frames** to get one vector per sample.

In [ ]:
def thermal_forward(encoder, images):
    # images: (batch, frames, 16, 16, 1)  — channels LAST
    n_frames = images.shape[1]
    # 🎯 (1) fold frames into batch AND move channels to the front
    x = rearrange(images, ??? )            # 🎯 input 'b f h w c'; on the right merge (b f) and put c after it
    feat = encoder(x)                      # (b*f, 32)
    feat = rearrange(feat, "(b f) d -> b f d", f=n_frames)
    # 🎯 (2) average the per-frame features into one vector per sample
    return reduce(feat, ??? )              # 🎯 drop the frames axis 'f' with a "mean"

# --- self-check ---
enc = basin_lab.VisionEncoder()
out = thermal_forward(enc, torch.zeros(2, 3, 16, 16, 1))
print("thermal branch output:", tuple(out.shape))
assert out.shape == (2, basin_lab.THERMAL_FEATURES), "Should be (2, 32) per-sample features."
print("✅ thermal branch outputs 32 features")

## 🎯 Block 3 — the MLP core, with its skip connection

After fusion, the features pass through a small **MLP core**. The plan draws it with a **`+`**
looping the input back onto the output — a **skip (residual) connection**: `out = relu(block(x) + x)`.
Skip connections let gradients flow straight through, so deeper stacks still train.

### 🧠 First, the shape constraint that the `+` imposes

In [ ]:
basin_viz.mc_quiz("skip_shape")

### 🎯 Write the core

Two `Linear` layers with a `ReLU` between them, then **add the input back** before a final `ReLU`.
Because of the skip, both layers must keep the width `d` (so `block(x)` and `x` line up).

In [ ]:
class MLPCore(nn.Module):
    def __init__(self, d):
        super().__init__()
        # 🎯 two Linear layers — both keep the width d (so the skip can add x back)
        self.lin1 = nn.Linear(d, d)
        self.lin2 = ???                    # 🎯 same in/out width as lin1

    def forward(self, x):
        h = torch.relu(self.lin1(x))
        h = self.lin2(h)
        # 🎯 add the input back (the skip), then a final ReLU
        return torch.relu( ??? )           # 🎯 the block's output plus the original input x

# --- self-check: a residual block must keep the shape it was given ---
core = MLPCore(8)
out = core(torch.zeros(4, 8))
print("core output:", tuple(out.shape))
assert out.shape == (4, 8), "A skip connection requires output shape == input shape."
print("✅ MLP core keeps its width — the skip lines up")

## 🎯 Block 4 — assemble the model (and fix the intern's two mistakes)

Now wire the branches together into the full `BasinNet`. The engineer's skeleton is below, but
**three things still need you** — two of them are mistakes an intern left behind:

- **Freeze the encoder (D).** It's marked *do not touch*; the optimizer must never update it.
- **The fusion width (E) — "the intern's *second* mistake".** The two branches output 32 features
  each and we `cat` them. What width must the MLP core expect?
- **The regression head (C) — "the intern's *first* mistake".** It was never added, so the model
  currently outputs a 64-feature vector instead of *one temperature*.

### 🧠 Settle the two intern mistakes first

In [ ]:
basin_viz.mc_quiz("fusion_dim")

In [ ]:
basin_viz.true_false_quiz("regression_head")

### 🧊 And the freeze

In [ ]:
basin_viz.mc_quiz("frozen_blocks")

### 🎯 Fill the three blanks

`forward` is already wired — it uses the `signal_extractor` and `thermal_forward` you built above.

In [ ]:
class BasinNet(nn.Module):
    def __init__(self, signal_extractor, encoder):
        super().__init__()
        self.signal_extractor = signal_extractor
        self.encoder = encoder

        # 🎯 D — keep the vision encoder FROZEN (optimizer must not touch its weights)
        for p in self.encoder.parameters():
            p.requires_grad_( ??? )        # 🎯 True or False?

        # 🎯 E — the fusion width: the two 32-feature branches, concatenated
        fusion_dim = ???                   # 🎯 add the two branch widths — basin_lab.SIGNAL_FEATURES and basin_lab.THERMAL_FEATURES
        self.core = MLPCore(fusion_dim)

        # 🎯 C — the regression head: fused features -> ONE temperature
        self.head = ???                    # 🎯 a Linear taking the fused width down to a single number

    def forward(self, x_sig, x_th):
        s = self.signal_extractor(x_sig)        # (B, 32)
        t = thermal_forward(self.encoder, x_th) # (B, 32)
        x = torch.cat([s, t], dim=1)            # (B, 64)
        return self.head(self.core(x)).squeeze(-1)   # (B,) one temperature per sample

# --- build it and self-check ---
encoder = basin_lab.VisionEncoder()
model = BasinNet(signal_extractor, encoder).to(device)

assert basin_lab.n_trainable(model.encoder) == 0, "The encoder must be frozen (0 trainable params)."
xb_sig, xb_th, yb = next(iter(train_loader))
pred = model(xb_sig.to(device), xb_th.to(device))
assert pred.shape == yb.shape, f"Expected one temperature per sample {tuple(yb.shape)}, got {tuple(pred.shape)}."
print("✅ encoder frozen, output is one temperature per sample:", tuple(pred.shape))
print("trainable parameters:", basin_lab.n_trainable(model))

---
# Part 4 — Train it and watch the basin temperature come into focus

Everything's wired. We train for a handful of epochs with **MSE loss** (it's a regression — we
penalise the squared error between predicted and true temperature).

> 📏 **The honest baseline.** The target is *standardized* (mean 0, variance 1), so a model that
> learned **nothing** and always predicts the mean scores a loss of **1.0**. We draw that line —
> a real curve has to drop clearly below it. (We only pass the **trainable** parameters to the
> optimizer, so the frozen encoder stays untouched.)

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3)

train_hist, val_hist = [], []
for epoch in range(15):
    tr_loss, _ = basin_lab.run_epoch(model, train_loader, optimizer=optimizer, device=device)
    va_loss, va_rmse = basin_lab.run_epoch(model, val_loader, device=device)   # no optimizer -> eval
    train_hist.append(tr_loss); val_hist.append(va_loss)

plt.plot(train_hist, label="train", marker="o")
plt.plot(val_hist, label="val", marker="o")
plt.axhline(1.0, ls="--", color="gray", lw=1, label="predict-the-mean baseline (loss = 1.0)")
plt.ylim(0, 1.15)
plt.xlabel("epoch"); plt.ylabel("MSE loss (standardized)"); plt.legend()
plt.title("Basin-temperature predictor — loss curves"); plt.show()
print(f"final validation RMSE: {va_rmse:.3f} (standardized) — baseline is 1.0")

**The loss dives well below the baseline** — the predictor learned to read both the sensor
trends and the thermal hot-spot. Now see it in the units that matter: map predictions back to **°C**
and check them against the **overheat line**.

In [ ]:
pred_std, true_std = basin_lab.predict_all(model, val_loader, device=device)
basin_viz.prediction_plot(pred_std, true_std,
                          bundle["y_mean"], bundle["y_std"], bundle["overheat_c"])

Points hug the diagonal, and crucially the model rarely lands on the *wrong side* of the red
overheat line — exactly the call the manager needs the backup system to get right.

---
# 🎓 Wrap-up — what you finished

You never redesigned the model. You **read the plan** and completed each block at the source:

| Block | What was wrong / missing | The fix |
|---|---|---|
| 🔌 Signal extractor | `SpectroNet` ended in a classifier, not features | `nn.Sequential(*children[:-1])` — drop the last child |
| 🖼️ Thermal reshape | frames were channels-last, with a frames axis | `rearrange` to `(b f) c h w`, encode, `reduce` over frames |
| ➕ MLP core | the skip `+` needs matching shapes | two `d→d` layers, then `relu(block(x) + x)` |
| 🧊 Freeze (D) | the "do-not-touch" encoder would have trained | `requires_grad_(False)` on its parameters |
| 📏 Fusion width (E) | intern hard-coded one branch's width | `32 + 32 = 64` after the `cat` |
| 🎯 Regression head (C) | no final layer → a vector, not a temperature | `nn.Linear(fusion_dim, 1)` |

**The throughline:** finishing an architecture is mostly **reading it carefully** — tracing shapes
across a `cat`, respecting what must stay frozen, and knowing that nesting/slicing `nn.Module`s lets
you reuse any part of a network without rewriting it.

> 🔌 *Plant integration:* export your finished architecture below and drop it into `workshop/solutions/`.
> The control panel rebuilds your model and **checks the architecture** — the right shapes, the frozen
> encoder, and that every block is wired in — **no training required**.

## 📦 Package your model for the control panel

The course ends with a **Nuclear Central Manager** control panel (in `workshop/`) that runs your
homework as a live power-plant. Part 2 is an **architecture** exercise, so the panel doesn't train
anything — it **rebuilds your model and checks it structurally**: the right shapes, a frozen
encoder, and that every block is wired into `forward`. Package your finished architecture as a
single `build_model()` so the panel can construct it.


In [ ]:
# 📦 Package your finished model as build_model() — the panel builds it fresh (no training).
def build_model():
    """Assemble a fresh BasinNet from the given building blocks (untrained)."""
    spectro = basin_lab.SpectroNet(in_channels=4)
    signal_extractor = nn.Sequential(*list(spectro.children())[:-1])   # drop the classifier
    encoder = basin_lab.VisionEncoder()
    return BasinNet(signal_extractor, encoder)

# self-check: it builds, ends in one temperature per sample, and keeps the encoder frozen
_m = build_model()
_out = _m(torch.zeros(2, 4, 64), torch.zeros(2, 3, 16, 16, 1))
assert _out.reshape(2, -1).shape == (2, 1), "build_model() must end in one temperature per sample."
assert sum(p.requires_grad for p in _m.encoder.parameters()) == 0, "The vision encoder must stay frozen."
print("build_model() ready ✅")


## 📦 Export your solution for the control panel

Run the cell below to write `solution_part2.py`, then drop it into `workshop/solutions/`. The panel
will rebuild **your** architecture and grade it against the same hidden structural checks (shapes,
frozen encoder, wiring) instead of the built-in reference.


In [ ]:
# 📦 Export YOUR finished architecture as solution_part2.py for the Nuclear Central Manager.
# This captures the *actual source* of the blocks you completed above
# (thermal_forward, MLPCore, BasinNet) plus build_model() — not a canned answer.
# Part 2 is graded structurally (no training, no weights), so only the architecture ships.
import inspect, os

HEADER = """# Part 2 solution — basin-temperature predictor architecture.
# Exported straight from the HW2 notebook: the blocks below are YOUR code.
import os, sys

import torch
import torch.nn as nn
from einops import rearrange, reduce

# Make the course basin helper importable from wherever this file is dropped.
for _rel in ("../homework/helpers", "../../workshop/homework/helpers",
             "helpers", "../../exercises", "../exercises", "exercises", "."):
    _cand = os.path.abspath(os.path.join(os.path.dirname(__file__), _rel))
    if os.path.exists(os.path.join(_cand, "basin_lab.py")):
        sys.path.insert(0, _cand)
        break
import basin_lab  # noqa: E402  (path injected just above)
"""

WRAPPER = """

class BasinSolution:
    def build_model(self):
        return build_model()


def get_solution():
    return BasinSolution()
"""

# Grab the blocks exactly as you defined them in this notebook.
try:
    bodies = [inspect.getsource(obj) for obj in (thermal_forward, MLPCore, BasinNet, build_model)]
except NameError as exc:
    raise NameError(
        "Run the cells that define thermal_forward, MLPCore, BasinNet and build_model "
        "(and re-run them after any edit) before exporting."
    ) from exc

source = HEADER + "\n" + "\n\n".join(b.rstrip() + "\n" for b in bodies) + WRAPPER

# Write next to the control panel (workshop/solutions/) when reachable, plus a local copy.
targets = []
panel = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "solutions", "solution_part2.py"))
if os.path.isdir(os.path.dirname(panel)):
    targets.append(panel)
targets.append(os.path.abspath(os.path.join(os.getcwd(), "solution_part2.py")))

for dest in dict.fromkeys(targets):  # de-dup while keeping order
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "w") as f:
        f.write(source)
    print("Wrote", dest)

print("\nDrop solution_part2.py into workshop/solutions/ (already done if the path above is there),")
print("then relaunch the control panel — it will grade YOUR architecture.")

try:  # in Colab, also offer the file as a download
    from google.colab import files
    files.download("solution_part2.py")
except Exception:
    pass
